### Setup

In [0]:
print("silver_started")

from pyspark.sql import functions as f
from pyspark.sql.window import Window

### Use Catalog & Schema

In [0]:
%sql
USE CATALOG marketing_analytics_capstone;
USE SCHEMA silver;

### Read Bronze

In [0]:
BRONZE_PATH="s3://marketing-analytics-capstone/bronze/" 
CHECKPOINTS_PATH = "s3://marketing-analytics-capstone/checkpoints/silver"  
SILVER_PATH='s3://marketing-analytics-capstone/silver'

In [0]:
df = (
    spark.readStream
    .table('marketing_analytics_capstone.bronze.bronze_campaigns')
    # .format("cloudFiles")
    # .option("cloudFiles.format", "parquet")   # or csv/parquet
    # .option("cloudFiles.schemaLocation", f"{CHECKPOINTS_PATH}/campaigns/schema/")
    # .option("cloudFiles.inferColumnTypes", "true")
    # .load(BRONZE_PATH)
)



### Standardize Column Names

In [0]:
df = (
    df
    .withColumnRenamed("Campaign_ID", "campaign_id")
    .withColumnRenamed("Campaign_Type", "campaign_type")
    .withColumnRenamed("Channel_Used", "channel_used")
    .withColumnRenamed("Clicks", "clicks")
    .withColumnRenamed("Company", "company")
    .withColumnRenamed("Conversion_Rate", "conversion_rate")
    .withColumnRenamed("Customer_Segment", "customer_segment")
    .withColumnRenamed("Date", "date_raw")
    .withColumnRenamed("Duration", "duration_raw")
    .withColumnRenamed("Engagement_Score", "engagement_score")
    .withColumnRenamed("Impressions", "impressions")
    .withColumnRenamed("Language", "language")
    .withColumnRenamed("Location", "location")
    .withColumnRenamed("ROI", "roi")
    .withColumnRenamed("Target_Audience", "target_audience")
)

### Safe Type Conversions (DEFENSIVE)

In [0]:
df = (
    df

    # Acquisition Cost → DOUBLE
    .withColumn(
        "acquisition_cost",
        f.when(
            f.regexp_replace("Acquisition_Cost", "[$,]", "").rlike("^[0-9.]+$"),
            f.regexp_replace("Acquisition_Cost", "[$,]", "").cast("double")
        )
    )

    # Date → DATE
    .withColumn(
    "date",
    f.to_timestamp("date_raw", "yyyy-MM-dd")
)

    # Duration → INT (days)
    .withColumn(
        "duration_days",
        f.when(
            f.col("duration_raw").rlike(r"\d+"),
            f.regexp_extract("duration_raw", r"(\d+)", 1).cast("int")
        )
    )
)

### Standardize Categorical Columns

In [0]:
df = (
    df
    .withColumn("campaign_type", f.lower(f.trim("campaign_type")))
    .withColumn("channel_used", f.lower(f.trim("channel_used")))
    .withColumn("company", f.lower(f.trim("company")))
    .withColumn("customer_segment", f.lower(f.trim("customer_segment")))
    .withColumn("language", f.lower(f.trim("language")))
    .withColumn("location", f.lower(f.trim("location")))
    .withColumn("target_audience", f.lower(f.trim("target_audience")))
)

### Derived Metrics

In [0]:
df = df.withColumn(
    "ctr",
    f.when(f.col("impressions") > 0,
           f.col("clicks") / f.col("impressions"))
)

### Deduplication

In [0]:
# df = df.withWatermark("date", "1 day")

# window = Window.partitionBy("campaign_id", "date").orderBy(f.col("ingestion_timestamp").desc())

# df = df.withColumn("row_num", f.row_number().over(window)) \
#        .filter("row_num = 1") \
#        .drop("row_num")

### Data Quality Flags

In [0]:
df = (
    df
    .withColumn("dq_valid_cost",
                f.col("acquisition_cost").isNotNull() & (f.col("acquisition_cost") > 0))
    .withColumn("dq_valid_date",
                f.col("date").isNotNull())
    .withColumn("dq_valid_impressions",
                f.col("impressions") > 0)
    .withColumn("dq_valid_clicks",
                (f.col("clicks") >= 0) & (f.col("clicks") <= f.col("impressions")))
    .withColumn("dq_valid_conversion_rate",
                (f.col("conversion_rate") >= 0) & (f.col("conversion_rate") <= 1))
)

### Overall DQ Flag

In [0]:
df = df.withColumn(
    "dq_pass",
    f.col("dq_valid_cost") &
    f.col("dq_valid_date") &
    f.col("dq_valid_impressions") &
    f.col("dq_valid_clicks") &
    f.col("dq_valid_conversion_rate")
)

### Cleanup Columns

In [0]:
df = df.drop(
    "date_raw",
    "duration_raw",
    "_rescued_data"
)

### Write to Silver

In [0]:
querey=(
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINTS_PATH}/campaigns/checkpoints/")
    .partitionBy("source_month")
    .trigger(availableNow=True)
    .option('path',f"{SILVER_PATH}/campaigns/")
    .toTable("marketing_analytics_capstone.silver.silver_campaigns")
)

querey.awaitTermination()

In [0]:
from delta.tables import DeltaTable

df = spark.read.table("marketing_analytics_capstone.silver.silver_campaigns")

window = Window.partitionBy("campaign_id", "date") \
    .orderBy(f.col("ingestion_timestamp").desc())

final_updates = (
    df.withColumn("rn", f.row_number().over(window))
      .filter("rn = 1")
      .drop("rn")
)

target = DeltaTable.forName(
    spark,
    "marketing_analytics_capstone.silver.silver_campaigns"
)

target.alias("t").merge(
    final_updates.alias("s"),
    "t.campaign_id = s.campaign_id AND t.date = s.date"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

### Validation Check

In [0]:
%sql
DESCRIBE DETAIL marketing_analytics_capstone.silver.silver_campaigns;

In [0]:
%sql
select count(*) from marketing_analytics_capstone.silver.silver_campaigns;